## Lab 2 Regression Case Study
#### Data Science UAB Summer Camp 2026


#### Predicting Medical Insurance Costs

In this lab, we will use a real dataset to study **regression**, a common machine learning method for predicting numerical outcomes.

Our main question is:

> Can we predict a person's annual medical insurance charges using information such as age, BMI, smoking status, and region?

This lab connects directly to the machine learning lecture. We will practice:

- Exploratory data analysis
- Data visualization
- Simple linear regression
- Multiple regression
- Prediction
- Training and testing
- Model assessment
- Communication of results

In [ ]:
if (!dir.exists("data")) {
  dir.create("data")
}

In [ ]:
library(data.table)

insurance <- fread(
  "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
)

head(insurance)

In [ ]:
fwrite(insurance, "data/insurance.csv")

#### Learning Goals

By the end of this lab, you should be able to:

1. Explain regression as a prediction method for numerical outcomes.
2. Identify response variables and predictor variables.
3. Create exploratory visualizations.
4. Fit simple and multiple regression models in R.
5. Use a fitted model to make predictions.
6. Split data into training and testing sets.
7. Evaluate model performance using RMSE.
8. Compare several regression models and choose a better one.

### <div class="alert alert-block alert-success"> Section 1: Business Understanding and Data Exploration </div>

Before writing code, we need to understand the problem.

Insurance companies need to estimate future medical costs. If predicted costs are too low, the company may lose money. If predicted costs are too high, customers may be charged too much.

In this dataset, each row represents one person.

The response variable is:

- `charges`: annual medical insurance charges

Possible predictors include:

- `age`
- `sex`
- `bmi`
- `children`
- `smoker`
- `region`

### 1.1 Load Packages

We will use:

- `data.table` for data import and manipulation
- `ggplot2` for visualization
- `dplyr` for readable data operations

In [ ]:
# Install packages if needed:
# install.packages("data.table")
# install.packages("ggplot2")
# install.packages("dplyr")
# install.packages("corrplot")

library(data.table)
library(ggplot2)
library(dplyr)

### 1.2 Load the Dataset

The dataset is stored in the `data/` folder.

In [ ]:
insurance <- fread("data/insurance.csv")

### 1.3 First Look at the Data

Whenever we receive a new dataset, we should first inspect it.

In [ ]:
head(insurance)

In [ ]:
dim(insurance)

In [ ]:
names(insurance)

In [ ]:
str(insurance)

In [ ]:
summary(insurance)

### Discussion Questions

1. How many observations are in the dataset?
2. How many variables are in the dataset?
3. Which variable are we trying to predict?
4. Which variables do you think may influence medical charges?

### 1.4 Check Missing Values

Real datasets often contain missing values. Let us check whether this dataset has any.

In [ ]:
colSums(is.na(insurance))

This dataset is relatively clean. That allows us to focus on regression modeling and prediction.

### <div class="alert alert-block alert-success"> Section 2: Exploratory Data Analysis </div>

Before building a model, we should explore the data.

Exploratory data analysis helps us understand:

- distributions
- relationships between variables
- unusual observations
- possible predictors

#### <div class="alert alert-block alert-success"> 2.1 Distribution of Medical Charges </div>

First, let us examine the response variable, `charges`.

In [ ]:
library(data.table)
library(ggplot2)

If you have trouble to set your working directory, please try

In [ ]:
insurance <- fread("https://raw.githubusercontent.com/kerenli/Data-Science-UAB-Summer-Camp/main/data/insurance.csv")

str(insurance)

or

In [ ]:
insurance <- fread("data/insurance.csv")

str(insurance)

In [ ]:
ggplot(insurance, aes(x = charges)) +
  geom_histogram(bins = 30, color = "white") +
  labs(
    title = "Distribution of Medical Insurance Charges",
    x = "Charges",
    y = "Count"
  ) +
  theme_minimal(base_size = 14)

### Discussion Questions

1. Are medical charges evenly distributed?
2. Do most people have low, medium, or high charges?
3. Are there people with extremely high charges?

#### <div class="alert alert-block alert-success"> 2.2 Charges vs Age </div>

Age may be related to medical costs. Let us visualize the relationship.

In [ ]:
ggplot(insurance, aes(x = age, y = charges)) +
  geom_point(alpha = 0.5) +
  #geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "Medical Charges vs Age",
    x = "Age",
    y = "Charges"
  ) +
  theme_minimal(base_size = 14)

#### <div class="alert alert-block alert-success"> 2.3 Charges vs BMI </div>

BMI is another possible predictor.

In [ ]:
ggplot(insurance, aes(x = bmi, y = charges)) +
  geom_point(alpha = 0.5) +
 # geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "Medical Charges vs BMI",
    x = "BMI",
    y = "Charges"
  ) +
  theme_minimal(base_size = 14)

#### <div class="alert alert-block alert-success"> 2.4 Charges by Smoking Status </div>

Smoking status is a categorical variable. A boxplot is useful for comparing charges between smokers and non-smokers.

In [ ]:
ggplot(insurance, aes(x = smoker, y = charges)) +
  geom_boxplot() +
  labs(
    title = "Medical Charges by Smoking Status",
    x = "Smoker",
    y = "Charges"
  ) +
  theme_minimal(base_size = 14)

### Discussion Questions

1. Which group has higher medical charges?
2. Is the difference small or large?
3. Why might smoking status be important for prediction?

#### <div class="alert alert-block alert-success"> 2.5 Charges by Region </div>

Region is another categorical variable.

In [ ]:
ggplot(insurance, aes(x = region, y = charges)) +
  geom_boxplot() +
  labs(
    title = "Medical Charges by Region",
    x = "Region",
    y = "Charges"
  ) +
  theme_minimal(base_size = 14)

#### <div class="alert alert-block alert-success"> 2.6 Correlation Among Numerical Variables </div>

Correlation measures the strength of a linear relationship between two numerical variables.

In [ ]:
num_vars <- insurance[ , .(age, bmi, children, charges)]

cor(num_vars)

Optional: If the `corrplot` package is installed, we can visualize the correlation matrix.

In [ ]:
# Optional visualization
library(corrplot)
corrplot(
  cor(num_vars),
  method = "circle"
)

### <div class="alert alert-block alert-success"> Section 3: Simple Linear Regression </div>

Regression is used when we want to predict a numerical outcome.

Here, we want to predict:

```text
charges
```

First, we will use only one predictor:

```text
age
```

This model asks:

> Can age alone help predict medical charges?

#### <div class="alert alert-block alert-success"> 3.1 Fit a Simple Linear Regression Model </div>

In [ ]:
fit_age <- lm(charges ~ age, data = insurance)

summary(fit_age)

#### <div class="alert alert-block alert-success"> 3.2 Interpreting the Model </div>

The model has the form:

```text
Predicted charges = intercept + slope × age
```

Questions:

1. Is the slope for age positive or negative?
2. Does the model suggest that older people tend to have higher charges?
3. Is age alone enough to explain medical charges?

#### <div class="alert alert-block alert-success"> 3.3 Make a Prediction </div>

Let us predict charges for a 40-year-old person.

In [ ]:
new_person <- data.frame(age = 40)

predict(
  fit_age,
  newdata = new_person
)

#### <div class="alert alert-block alert-success"> 3.4 Visualize the Simple Regression Model </div>

In [ ]:
ggplot(insurance, aes(x = age, y = charges)) +
  geom_point(alpha = 0.4) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(
    title = "Simple Regression: Predicting Charges from Age",
    x = "Age",
    y = "Charges"
  ) +
  theme_minimal(base_size = 14)

### <div class="alert alert-block alert-success"> Section 4: Multiple Regression  </div>

Age alone does not capture the whole story.

Medical charges may depend on several variables at the same time.

Multiple regression allows us to use several predictors.

#### <div class="alert alert-block alert-success"> 4.1 Model with Age and BMI  </div>

In [ ]:
fit_age_bmi <- lm(charges ~ age + bmi, data = insurance)

summary(fit_age_bmi)

#### <div class="alert alert-block alert-success"> 4.2 Model with Age, BMI, and Smoking Status  </div>

Now we add `smoker`, which appeared important in the exploratory analysis.

In [ ]:
fit_age_bmi_smoker <- lm(charges ~ age + bmi + smoker, data = insurance)

summary(fit_age_bmi_smoker)

### Discussion Questions

1. What happens after adding `smoker`?
2. Does the model fit improve?
3. How would you explain the smoker coefficient in words?

Important idea:

> A regression model can include both numerical and categorical predictors.

#### <div class="alert alert-block alert-success"> 4.3 Full Model </div>

Now we include all available predictors.

In [ ]:
fit_full <- lm(charges ~ ., data = insurance)

summary(fit_full)

#### <div class="alert alert-block alert-success"> 4.4 Compare Model Summaries </div>

The `summary()` output includes a value called **R-squared**.

R-squared is a measure of how much variation in the response is explained by the model.

Larger R-squared usually means the model explains more variation, but this does not always mean the model will predict better on new data.

In [ ]:
summary(fit_age)$r.squared
summary(fit_age_bmi)$r.squared
summary(fit_age_bmi_smoker)$r.squared
summary(fit_full)$r.squared

#### Discussion

1. Which model has the largest R-squared?
2. Does adding variables always increase R-squared?
3. Why do we still need testing data?

### <div class="alert alert-block alert-success"> Section 5: Training and Testing </div>

In machine learning, we do not only care about how well a model fits the data it has already seen.

We care about whether the model can predict new observations.

To test this, we split the dataset into:

- **Training data**: used to fit the model
- **Testing data**: used to evaluate prediction performance

#### <div class="alert alert-block alert-success"> 5.1 Create a Train/Test Split  </div>

In [ ]:
set.seed(123)

n <- nrow(insurance)

train_id <- sample(1:n, size = round(0.8 * n))

train <- insurance[train_id]
test  <- insurance[-train_id]

nrow(train)
nrow(test)

#### <div class="alert alert-block alert-success"> 5.2 Fit Models Using Training Data Only </div>

In [ ]:
fit_train_age <- lm(charges ~ age, data = train)

fit_train_age_bmi <- lm(charges ~ age + bmi, data = train)

fit_train_age_bmi_smoker <- lm(charges ~ age + bmi + smoker, data = train)

fit_train_full <- lm(charges ~ ., data = train)

#### <div class="alert alert-block alert-success"> 5.3 Predict on the Testing Data </div>

In [ ]:
pred_age <- predict(fit_train_age, newdata = test)

pred_age_bmi <- predict(fit_train_age_bmi, newdata = test)

pred_age_bmi_smoker <- predict(fit_train_age_bmi_smoker, newdata = test)

pred_full <- predict(fit_train_full, newdata = test)

### <div class="alert alert-block alert-success"> Section 6: Model Assessment </div>

We need a way to measure prediction error.

A common measure for regression is **RMSE**, which stands for:

```text
Root Mean Squared Error
```

RMSE measures the typical size of prediction errors.

Smaller RMSE means better prediction performance.

#### <div class="alert alert-block alert-success"> 6.1 Define an RMSE Function </div>

In [ ]:
rmse <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2))
}

#### <div class="alert alert-block alert-success"> 6.2 Compare Test RMSE Across Models </div>

In [ ]:
model_results <- data.table(
  model = c(
    "Age only",
    "Age + BMI",
    "Age + BMI + Smoker",
    "Full model"
  ),
  test_rmse = c(
    rmse(test$charges, pred_age),
    rmse(test$charges, pred_age_bmi),
    rmse(test$charges, pred_age_bmi_smoker),
    rmse(test$charges, pred_full)
  )
)

model_results

#### <div class="alert alert-block alert-success"> 6.3 Visualize Model Comparison </div>

In [ ]:
ggplot(model_results, aes(x = reorder(model, test_rmse), y = test_rmse)) +
  geom_col() +
  coord_flip() +
  labs(
    title = "Comparing Regression Models",
    x = "Model",
    y = "Test RMSE"
  ) +
  theme_minimal(base_size = 14)

### Discussion

1. Which model has the lowest test RMSE?
2. Did adding smoking status improve prediction?
3. Did the full model perform best?
4. What does RMSE mean in this problem?

#### <div class="alert alert-block alert-success"> 6.4 Actual vs Predicted Plot </div>

A good prediction model should produce predicted values close to actual values.

If predictions were perfect, all points would lie on the diagonal line.

In [ ]:
test_results <- copy(test)
test_results[, predicted_charges := pred_full]

max_charge <- max(test_results$charges, test_results$predicted_charges)

ggplot(test_results, aes(x = charges, y = predicted_charges)) +
  geom_point(alpha = 0.6) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed") +
  coord_equal(xlim = c(0, max_charge), ylim = c(0, max_charge)) +
  labs(
    title = "Actual vs Predicted Medical Charges",
    x = "Actual Charges",
    y = "Predicted Charges"
  ) +
  theme_minimal(base_size = 14)

### <div class="alert alert-block alert-success">  Section 7: Mini Model Competition </div>

Now it is your turn.

Work with a partner. Build at least two regression models and compare their testing RMSE.

Your goal:

> Find a model that predicts medical charges as accurately as possible.

#### <div class="alert alert-block alert-success"> 7.1 Build Your Own Models </div>

You may use any of the following predictors:

- `age`
- `sex`
- `bmi`
- `children`
- `smoker`
- `region`

Example model formulas:

```r
charges ~ age + bmi
charges ~ age + bmi + smoker
charges ~ age + bmi + smoker + children
charges ~ age + bmi + smoker + region
charges ~ age + bmi + smoker + children + region
```

In [ ]:
# Model A
fit_A <- lm(charges ~ age + bmi, data = train)

pred_A <- predict(fit_A, newdata = test)

rmse_A <- rmse(test$charges, pred_A)

rmse_A

In [ ]:
# Model B
fit_B <- lm(charges ~ age + bmi + smoker, data = train)

pred_B <- predict(fit_B, newdata = test)

rmse_B <- rmse(test$charges, pred_B)

rmse_B

#### <div class="alert alert-block alert-success"> 7.2 Record Your Results </div>

Fill in your model formulas and RMSE values.

| Model | Formula | Test RMSE |
|---|---|---|
| A |  |  |
| B |  |  |
| C |  |  |

Which model performed best?

#### <div class="alert alert-block alert-success"> 7.3 Interpret Your Best Model </div>

Answer the following questions:

1. Which predictors did you include?
2. Which predictor seemed most important?
3. How well did your model predict charges?
4. What surprised you about the dataset?

### <div class="alert alert-block alert-success"> Section 8: Short Assessment </div>

Answer the following questions individually.

1. What is the response variable in this lab?
2. What is one predictor used in the regression model?
3. Why do we split data into training and testing sets?
4. What does RMSE measure?
5. Which variable appeared to have a large impact on medical charges?
6. Why might a model that fits training data well still perform poorly on new data?

#### Lab 2 Summary

Today we used regression to predict medical insurance charges.

We learned that:

- Regression predicts numerical outcomes.
- Exploratory analysis helps identify useful predictors.
- Multiple regression can use both numerical and categorical predictors.
- Training data are used to fit a model.
- Testing data are used to evaluate prediction performance.
- RMSE measures typical prediction error.
- A good model should predict well on new data, not just training data.

In the next lab, we will study **classification**, where the response variable is categorical rather than numerical.